In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load the zip code geography data
gdf_zipcodecom = gpd.read_file('../api/src/migrations/data/20260227-zip-code-data.geojson')
gdf_top = gpd.read_file("../api/src/migrations/data/20251228-us_zcta_topo.json")

print("zipcodescom columns:", gdf_zipcodecom.columns.tolist())
print("zcta_top columns:", gdf_top.columns.tolist())
print(f"\nzipcodescom rows: {len(gdf_zipcodecom)}, zcta_top rows: {len(gdf_top)}")
print("\nzipcodescom sample:")
display(gdf_zipcodecom.head(3))
print("\nzcta_top sample:")
display(gdf_top.head(3))


In [ ]:
# Normalize zip code column names so we can join on them.
# Update these if the output above shows different column names.
ZIPCODECOM_ZIP_COL = 'ZIP'   # adjust if needed
ZCTA_TOP_ZIP_COL = 'zip'   # adjust if needed

gdf_zipcodecom['_zip'] = gdf_zipcodecom[ZIPCODECOM_ZIP_COL].astype(str).str.zfill(5)
gdf_top['_zip'] = gdf_top[ZCTA_TOP_ZIP_COL].astype(str).str.zfill(5)

shared_zips = set(gdf_zipcodecom['_zip']) & set(gdf_top['_zip'])
print(f"Zip codes in zipcodescom: {len(gdf_zipcodecom)}")
print(f"Zip codes in zcta_top:    {len(gdf_top)}")
print(f"Shared zip codes:         {len(shared_zips)}")


In [ ]:
# Sample 300 shared zip codes randomly and compute detail metrics for each.
SAMPLE_SIZE = 300
rng = np.random.default_rng(42)
sample_zips = rng.choice(sorted(shared_zips), size=min(SAMPLE_SIZE, len(shared_zips)), replace=False)

def count_vertices(geom):
    """Total coordinate count across all rings of a (Multi)Polygon."""
    if geom is None or geom.is_empty:
        return 0
    if geom.geom_type == 'Polygon':
        return sum(len(ring.coords) for ring in [geom.exterior] + list(geom.interiors))
    elif geom.geom_type == 'MultiPolygon':
        return sum(count_vertices(p) for p in geom.geoms)
    return 0

rows = []
for z in sample_zips:
    geom_com = gdf_zipcodecom.loc[gdf_zipcodecom['_zip'] == z, 'geometry'].iloc[0]
    geom_top = gdf_top.loc[gdf_top['_zip'] == z, 'geometry'].iloc[0]
    rows.append({
        'zip': z,
        'vertices_com': count_vertices(geom_com),
        'vertices_top': count_vertices(geom_top),
    })

df = pd.DataFrame(rows)
df['vertex_ratio'] = df['vertices_com'] / df['vertices_top'].replace(0, np.nan)

print(df[['zip', 'vertices_com', 'vertices_top', 'vertex_ratio']].describe())


In [ ]:
# Distribution of vertex counts — side-by-side histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['vertices_com'], bins=50, color='steelblue', alpha=0.8, label='zipcodescom')
axes[0].hist(df['vertices_top'], bins=50, color='coral', alpha=0.7, label='zcta_top')
axes[0].set_xlabel('Vertex count')
axes[0].set_ylabel('Number of zip codes')
axes[0].set_title('Vertex count distribution (linear scale)')
axes[0].legend()

axes[1].hist(np.log1p(df['vertices_com']), bins=50, color='steelblue', alpha=0.8, label='zipcodescom')
axes[1].hist(np.log1p(df['vertices_top']), bins=50, color='coral', alpha=0.7, label='zcta_top')
axes[1].set_xlabel('log(1 + vertex count)')
axes[1].set_ylabel('Number of zip codes')
axes[1].set_title('Vertex count distribution (log scale)')
axes[1].legend()

plt.suptitle(f'Polygon detail comparison — {len(df)} sampled shared zip codes', fontsize=13)
plt.tight_layout()
plt.show()

print("\nMedian vertices:")
print(f"  zipcodescom: {df['vertices_com'].median():.0f}")
print(f"  zcta_top:    {df['vertices_top'].median():.0f}")
print(f"\nMean vertex ratio (com / top): {df['vertex_ratio'].mean():.2f}x")
print(f"Median vertex ratio:           {df['vertex_ratio'].median():.2f}x")


In [ ]:
# Visual side-by-side geometry comparison for a handful of example zip codes.
# Pick 6 zips that span the range of vertex ratios.
examples = df.sort_values('vertex_ratio').iloc[np.linspace(0, len(df)-1, 6, dtype=int)][['zip', 'vertices_com', 'vertices_top', 'vertex_ratio']]
print("Selected example zip codes:")
display(examples)

fig, axes = plt.subplots(len(examples), 2, figsize=(10, len(examples) * 3))

for i, row in enumerate(examples.itertuples()):
    geom_com = gdf_zipcodecom.loc[gdf_zipcodecom['_zip'] == row.zip, 'geometry'].iloc[0]
    geom_top = gdf_top.loc[gdf_top['_zip'] == row.zip, 'geometry'].iloc[0]

    for ax, geom, label, vcount, color in [
        (axes[i, 0], geom_top, 'zcta_top', row.vertices_top, 'coral'),
        (axes[i, 1], geom_com, 'zipcodescom', row.vertices_com, 'steelblue'),
    ]:
        gpd.GeoSeries([geom]).plot(ax=ax, edgecolor=color, facecolor='none', linewidth=1.2)
        ax.set_title(f'ZIP {row.zip} — {label}\n{vcount} vertices', fontsize=9)
        ax.set_axis_off()

plt.suptitle('Shape comparison: zcta_top (left) vs zipcodescom (right)', fontsize=12)
plt.tight_layout()
plt.show()
